# Phase 4 — Advanced Data Quality & Monitoring (Google Colab)

**Objective:** Run your healthcare Bronze/Silver/Gold pipeline with independent, fail-closed validation, inspect structured metrics, and deliberately inject bad data to prove that the quality gates detect it.

**Architecture:** Generate Bronze → Build Silver/Gold → Quality gates → JSON metrics. The same three stages are orchestrated by Airflow when running on a MacBook.

This notebook uses **synthetic data only** and does not run an Airflow server. Airflow's UI is run through Docker on your MacBook.

## Step 1 — Install PySpark

The project uses Spark 3.5.x. Colab might show a warning that a preinstalled `dataproc-spark-connect` package expects 4.x; this is not necessarily a failure. Restart the session after installation when prompted.

In [ ]:
%pip -q install "pyspark>=3.5,<4"

## Step 2 — Get latest GitHub code

Start fresh so old modules are not cached, or restart the runtime. The notebook uses a new output directory for Phase 4.

In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT = Path("/content/healthcare-lakehouse-data-platform")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/mosesrathan19/healthcare-lakehouse-data-platform.git",
        str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(PROJECT))
OUTPUT = PROJECT / "outputs/phase4_colab_demo"
print("Project:", PROJECT)

## Step 3 — Generate Bronze

Generate synthetic patients, providers, encounters and claims. We use `demo` (10K encounters and claims) first to keep the quality tutorial manageable.

In [ ]:
from src.phase4_cli import run_stage
from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[*]")
    .appName("HealthcarePhase4Colab")
    .config("spark.sql.shuffle.partitions", "8").getOrCreate())
run_stage(spark, "generate", OUTPUT, scale="demo")

## Step 4 — Build Silver and Gold

This is the same validated lakehouse builder from Phase 1, now executed as a separate step so Airflow can track it independently.

In [ ]:
run_stage(spark, "build", OUTPUT, scale="demo")

## Step 5 — Run the advanced data-quality checks

The validator confirms required columns, primary-key uniqueness, foreign keys, nonnegative monetary values, valid claim statuses and dates, quarantine ratios, expected row counts, and Gold-versus-Silver reconciliation. It writes a structured JSON report.

In [ ]:
event = run_stage(spark, "validate", OUTPUT, scale="demo")
report = event["result"]
print("Status:", report["status"])
print("Checks:", len(report["checks"]))
print("Failed:", report["failed_checks"])
assert report["status"] == "PASS"

## Step 6 — Inspect the quality report

Each check includes a name, status, actual result, expected result, and severity. Airflow uses the same report to decide whether a pipeline should continue.

In [ ]:
import pandas as pd
from IPython.display import display
df = pd.DataFrame(report["checks"])
display(df)
print((OUTPUT / "monitoring" / "quality_report.json").read_text()[:4000])

## Step 7 — Inject a bad claim (without modifying your source files)

We create one extra fake claim referencing a nonexistent encounter, with a negative billed amount. We expect our quality engine to **FAIL**. This shows the rules are useful, rather than always passing.

In [ ]:
from src.data_quality import check_frames
from pyspark.sql import functions as F
names = ["patients", "providers", "encounters", "claims"]
frames = {name: spark.read.parquet(str(OUTPUT / "silver" / name)) for name in names}
bad = (frames["claims"].limit(1)
       .withColumn("claim_id", F.lit("CLM-BAD-EXAMPLE"))
       .withColumn("encounter_id", F.lit("ENC-NOT-FOUND"))
       .withColumn("billed_cents", F.lit(-100)))
corrupted = {**frames, "claims": frames["claims"].unionByName(bad)}
bad_report = check_frames(corrupted)
print("Expected failure:", bad_report["status"])
print("Detected:", bad_report["failed_checks"])
assert bad_report["status"] == "FAIL"
assert "claims.encounter_fk" in bad_report["failed_checks"]
assert "claims.nonnegative_amount" in bad_report["failed_checks"]

## Step 8 — Inspect monitoring artifacts

Each stage writes a latest-run event JSON with start/end UTC, elapsed seconds, Spark/Python version, status, and results. These are **local educational monitoring artifacts**, not a production observability platform.

In [ ]:
import json
for stage in ["generate", "build", "validate"]:
    info = json.loads((OUTPUT / "monitoring" / f"{stage}_latest.json").read_text())
    print(stage, "→", info["status"], "|", info["elapsed_seconds"], "seconds")

## Step 9 — Run tests (optional)

These are integration tests; Spark may run a few expensive jobs. The tests confirm both clean data passes and deliberately invalid data is rejected.

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pytest", "-q",
                "tests/test_data_quality.py"],
               cwd=PROJECT, check=True)

## What's next?

On your **MacBook**, run Docker Desktop, clone or pull this repository, and start Airflow using `airflow/compose.yaml`. Airflow then executes the same three stages with retries and task logs.

See `docs/PHASE4_QUALITY_AIRFLOW.md`. Colab is temporary; do not use it as a persistent Airflow scheduler.